# 🚗 Smart Chassis Vibration Analysis System
**Project Module-4 · Related topic: Chassis Structures & Safety Telltales**

**Objective:** Detect and analyze vibration in a car chassis to predict potential structural damage (cracks / loose joints) and warn the driver with a telltale.

**How this notebook maps to the project concept**

| Project concept | Notebook section |
|---|---|
| Accelerometers on key chassis points | 2. Simulated 3-sensor chassis data (front / mid / rear) |
| Microcontroller processes data, finds abnormal frequency patterns | 3. Feature extraction (FFT / PSD) + 4. ML classifier |
| Warning telltale if vibration exceeds threshold | 5. Threshold + debounce telltale logic (MCU-style) |
| Model explaining how vibration predicts structural issues | 6. Live drive simulation + 7. Summary |

> No hardware needed – vibration data is **simulated** using physically motivated signatures. Section 8 shows how to plug in real MPU6050/ADXL345 data.

**Physical idea used for the simulation**
- **Healthy chassis:** stable natural frequencies (modes) + road noise + engine harmonics.
- **Crack:** stiffness loss → natural frequencies **shift down**, damping drops → **higher amplitude**, nonlinear "breathing" crack adds **harmonics**.
- **Loose joint:** intermittent **rattling impacts** → spiky, high-frequency bursts → **high kurtosis / crest factor**.

In [ ]:
# 1. Setup
import numpy as np
import matplotlib.pyplot as plt
from scipy import signal, stats
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay

rng = np.random.default_rng(42)

FS = 1000            # sampling rate (Hz) - typical for MEMS accelerometer
WIN = 1.0            # analysis window (seconds)
N = int(FS * WIN)
SENSORS = ["Front", "Middle", "Rear"]
CLASSES = ["Healthy", "Crack", "Loose joint"]
plt.rcParams["figure.figsize"] = (10, 4)
print("Setup done.")

## 2. Simulating accelerometer data on the chassis
Three accelerometers (front / middle / rear cross-members). Severity `0…1` controls how advanced the fault is.

In [ ]:
BASE_MODES = np.array([18.0, 45.0, 90.0])   # chassis natural frequencies (Hz) - illustrative
BASE_AMPS  = np.array([1.0, 0.7, 0.4])      # modal amplitudes (g-ish units)

def road_noise(n):
    b, a = signal.butter(4, 120 / (FS / 2))       # road input is mostly < 120 Hz
    x = signal.lfilter(b, a, rng.normal(0, 1, n + 200))[200:]
    return 0.5 * x / x.std()

def simulate_window(condition="Healthy", severity=0.0, gains=(1.0, 0.8, 1.1)):
    # Returns array (3 sensors, N samples) for one window.
    t = np.arange(N) / FS
    out = []
    # random operating condition (engine speed -> harmonic frequency)
    eng = rng.uniform(22, 30)
    for g in gains:
        modes = BASE_MODES * (1 + rng.normal(0, 0.01, 3))     # small natural scatter
        amps = BASE_AMPS * rng.uniform(0.85, 1.15, 3)
        sig = np.zeros(N)

        if condition == "Crack":
            modes = modes * (1 - 0.15 * severity)             # stiffness loss -> freq drop
            amps = amps * (1 + 0.9 * severity)                # less damping -> larger response
        for f, A in zip(modes, amps):
            sig += A * np.sin(2 * np.pi * f * t + rng.uniform(0, 2 * np.pi))
        if condition == "Crack":                              # breathing-crack harmonics
            sig += 0.45 * severity * np.sin(2 * np.pi * 2 * modes[0] * t + rng.uniform(0, 6.28))
            sig += 0.35 * severity * np.sin(2 * np.pi * 3 * modes[0] * t + rng.uniform(0, 6.28))

        # engine harmonics + road noise (present in all cases)
        sig += 0.5 * np.sin(2 * np.pi * eng * t) + 0.25 * np.sin(2 * np.pi * 2 * eng * t)
        sig += road_noise(N)

        if condition == "Loose joint":                        # rattle impacts
            n_hits = int(2 + 10 * severity)
            for _ in range(n_hits):
                i0 = rng.integers(0, N - 60)
                fr = rng.uniform(150, 250)
                tt = np.arange(60) / FS
                sig[i0:i0 + 60] += (3.0 * severity + 1.0) * np.exp(-tt * 250) * np.sin(2 * np.pi * fr * tt)

        sig = g * sig + rng.normal(0, 0.05, N)                # sensor noise
        out.append(sig)
    return np.array(out)

# Plot one example of each condition (front sensor)
fig, ax = plt.subplots(3, 2, figsize=(12, 8))
for r, cond in enumerate(CLASSES):
    x = simulate_window(cond, 0.9 if cond != "Healthy" else 0)[0]
    t = np.arange(N) / FS
    ax[r, 0].plot(t, x, lw=0.7); ax[r, 0].set_title(f"{cond} - time domain"); ax[r, 0].set_ylabel("accel")
    f, P = signal.welch(x, FS, nperseg=256)
    ax[r, 1].semilogy(f, P); ax[r, 1].set_title(f"{cond} - power spectrum"); ax[r, 1].set_xlim(0, 300)
ax[2, 0].set_xlabel("time (s)"); ax[2, 1].set_xlabel("frequency (Hz)")
plt.tight_layout(); plt.show()

## 3. Feature extraction (what the microcontroller would compute)
Per sensor and per 1-second window:
- **RMS** – overall vibration level
- **Peak / Crest factor** – spikiness
- **Kurtosis** – impulsiveness (loose joints → very high)
- **Dominant frequency** and **spectral centroid** – shifts indicate stiffness change (cracks)
- **Band energies** (low 5–30 Hz, mid 30–70 Hz, high 70–300 Hz)

In [ ]:
FEATURE_NAMES = ["rms", "peak", "crest", "kurtosis", "dom_freq", "centroid", "band_low", "band_mid", "band_high"]

def channel_features(x):
    x = x - x.mean()
    rms = np.sqrt(np.mean(x ** 2))
    peak = np.max(np.abs(x))
    f, P = signal.welch(x, FS, nperseg=256)
    m = (f >= 5) & (f <= 300)
    f, P = f[m], P[m]
    dom = f[np.argmax(P)]
    centroid = np.sum(f * P) / np.sum(P)
    def band(lo, hi):
        mm = (f >= lo) & (f < hi)
        return np.sum(P[mm]) / np.sum(P)
    return [rms, peak, peak / rms, stats.kurtosis(x), dom, centroid, band(5, 30), band(30, 70), band(70, 300)]

def window_features(w):
    # w: (3, N) -> flat vector of 3 x 9 features
    return np.concatenate([channel_features(ch) for ch in w])

ALL_NAMES = [f"{s[0]}_{n}" for s in SENSORS for n in FEATURE_NAMES]

# Build dataset
X, y = [], []
per_class = 300
for ci, cond in enumerate(CLASSES):
    for _ in range(per_class):
        sev = 0.0 if cond == "Healthy" else rng.uniform(0.3, 1.0)
        X.append(window_features(simulate_window(cond, sev)))
        y.append(ci)
X, y = np.array(X), np.array(y)
print("Dataset:", X.shape, "-> samples x features")

## 4. Machine-learning classifier (detect *and* identify the fault type)

In [ ]:
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.25, stratify=y, random_state=0)
clf = RandomForestClassifier(n_estimators=200, random_state=0)
clf.fit(X_tr, y_tr)
pred = clf.predict(X_te)

print(classification_report(y_te, pred, target_names=CLASSES))
ConfusionMatrixDisplay(confusion_matrix(y_te, pred), display_labels=CLASSES).plot(cmap="Blues")
plt.title("Confusion matrix (test set)"); plt.show()

imp = clf.feature_importances_
idx = np.argsort(imp)[::-1][:10]
plt.barh([ALL_NAMES[i] for i in idx][::-1], imp[idx][::-1])
plt.title("Top 10 features - which vibration indicators matter most?"); plt.show()

## 5. Telltale warning logic (threshold + debounce, MCU-style)
A real microcontroller wouldn't run a random forest – it would use a simple, robust rule. We learn a **healthy baseline** and compute an **anomaly score** (largest z-score among RMS, kurtosis, and dominant-frequency shift). The telltale turns **ON** only after the score exceeds the threshold for `N_ON` consecutive windows (avoids false alarms from a single pothole) and turns **OFF** after `N_OFF` normal windows (hysteresis).

In [ ]:
healthy = X[y == 0]
def cols(name): return [ALL_NAMES.index(f"{s[0]}_{name}") for s in SENSORS]

def indicator(feat, nm):
    v = feat[..., cols(nm)]
    return v.max(axis=-1) if nm == "kurtosis" else v.mean(axis=-1)   # kurtosis: worst sensor

base = {}
for nm in ["rms", "kurtosis", "dom_freq"]:
    v = indicator(healthy, nm)
    base[nm] = (v.mean(), max(v.std(), 4.0 if nm == "dom_freq" else 1e-3))   # floor std = FFT bin resolution

def anomaly_score(feat):
    # largest z-score among: RMS up, kurtosis up (impulsiveness), dominant-frequency shift
    z = []
    for nm in ["rms", "kurtosis", "dom_freq"]:
        d = (indicator(feat, nm) - base[nm][0]) / base[nm][1]
        z.append(abs(d) if nm == "dom_freq" else d)
    return max(z)

THRESH = 4.0    # warning threshold (in standard deviations from healthy baseline)
N_ON, N_OFF = 3, 5

class Telltale:
    def __init__(self):
        self.on, self.hi, self.lo = False, 0, 0
    def update(self, score):
        if score > THRESH: self.hi += 1; self.lo = 0
        else: self.lo += 1; self.hi = 0
        if not self.on and self.hi >= N_ON: self.on = True
        if self.on and self.lo >= N_OFF: self.on = False
        return self.on

hs = [anomaly_score(f) for f in healthy]
print(f"Healthy anomaly score: mean={np.mean(hs):.2f}, 99th pct={np.percentile(hs,99):.2f} (threshold={THRESH})")

## 6. Drive simulation: fault develops gradually while the system monitors

In [ ]:
# ---- choose scenario: "Crack" or "Loose joint" ----
SCENARIO = "Crack"
DURATION = 90        # seconds (one window per second)
FAULT_START = 30     # fault begins here and grows to full severity

times = np.arange(DURATION)
tt = Telltale()
rows = []
for k in times:
    if k < FAULT_START:
        cond, sev = "Healthy", 0.0
    else:
        cond, sev = SCENARIO, min(1.0, 0.15 + (k - FAULT_START) / (DURATION - FAULT_START))
    feat = window_features(simulate_window(cond, sev))
    sc = anomaly_score(feat)
    led = tt.update(sc)
    proba = clf.predict_proba(feat.reshape(1, -1))[0]
    rows.append((sev, sc, led, proba, feat[cols("rms")].mean(), feat[cols("kurtosis")].mean()))

sev_arr = np.array([r[0] for r in rows]); score = np.array([r[1] for r in rows])
led = np.array([r[2] for r in rows]); proba = np.array([r[3] for r in rows])

fig, ax = plt.subplots(3, 1, figsize=(11, 9), sharex=True)
ax[0].plot(times, sev_arr, color="gray"); ax[0].set_ylabel("true fault severity"); ax[0].set_title(f"Scenario: {SCENARIO} developing after t={FAULT_START}s")
ax[1].plot(times, score, label="anomaly score"); ax[1].axhline(THRESH, color="r", ls="--", label="threshold")
ax[1].fill_between(times, 0, score.max() * 1.05, where=led, color="red", alpha=0.15, label="TELLTALE ON")
ax[1].set_ylabel("anomaly score"); ax[1].legend()
for i, c in enumerate(CLASSES): ax[2].plot(times, proba[:, i], label=c)
ax[2].set_ylabel("ML probability"); ax[2].set_xlabel("time (s)"); ax[2].legend()
plt.tight_layout(); plt.show()

first_on = np.argmax(led) if led.any() else None
if first_on is not None:
    print(f"Telltale first lit at t={first_on}s -> severity at that moment ~{sev_arr[first_on]:.2f}")
    print("Dashboard:  \U0001F534 CHECK CHASSIS" )
else:
    print("Telltale never triggered - try lowering THRESH.")

### Dashboard-style telltale (LED) view

In [ ]:
def show_dashboard(k):
    on = led[k]
    plt.figure(figsize=(4, 2.2))
    plt.gca().add_patch(plt.Circle((0.5, 0.5), 0.35, color="red" if on else "green"))
    plt.text(0.5, 0.5, "CHECK\nCHASSIS" if on else "OK", ha="center", va="center", color="white", weight="bold")
    plt.title(f"t = {k}s   score={score[k]:.1f}"); plt.xlim(0, 1); plt.ylim(0, 1); plt.axis("off"); plt.show()

show_dashboard(FAULT_START - 5)                       # before fault
show_dashboard(min(DURATION - 1, FAULT_START + 40))   # fault developed

## 7. Sensitivity study – how small a fault can we catch?

In [ ]:
sevs = np.linspace(0.05, 1.0, 12)
res = {}
for cond in ["Crack", "Loose joint"]:
    res[cond] = []
    for s in sevs:
        det = np.mean([anomaly_score(window_features(simulate_window(cond, s))) > THRESH for _ in range(40)])
        res[cond].append(det)
for cond in res: plt.plot(sevs, res[cond], marker="o", label=cond)
plt.xlabel("fault severity"); plt.ylabel("detection rate (window > threshold)"); plt.title("Detection rate vs. severity")
plt.legend(); plt.grid(alpha=0.3); plt.show()

## 8. Using real hardware data (optional)
Wire an **MPU6050 / ADXL345** to an **Arduino/ESP32**, sample at ~1 kHz, log CSV `t,ax,ay,az`, upload to Colab and use:
```python
import pandas as pd
df = pd.read_csv("chassis_log.csv")          # columns: t, ax, ay, az
x = df["az"].values
fs = 1 / np.mean(np.diff(df["t"]))
# cut into 1-second windows, then reuse channel_features() / anomaly_score()
```

**Arduino-style pseudo-code for the telltale (mirrors Section 5):**
```cpp
const float THRESH = 4.0; int hi = 0, lo = 0; bool led = false;
void loop() {
  float rms = computeRMS(1000);          // 1 s of samples
  float kurt = computeKurtosis();
  float dom = dominantFreqFFT();
  float score = max(abs(rms-rmsBase)/rmsStd, max(abs(kurt-kBase)/kStd, abs(dom-fBase)/fStd));
  if (score > THRESH) { hi++; lo = 0; } else { lo++; hi = 0; }
  if (!led && hi >= 3) led = true;
  if (led && lo >= 5) led = false;
  digitalWrite(LED_PIN, led);
}
```

## 9. Summary / Expected outcome
1. **Sensors** at key chassis points capture vibration continuously.
2. **Frequency-domain features** reveal structural change: cracks lower natural frequency and add harmonics/amplitude; loose joints add impulsive, high-kurtosis rattle.
3. **A baseline + threshold with debounce** gives a simple, MCU-friendly rule that lights the **telltale** before the damage becomes severe.
4. **ML** (optional) additionally identifies *which* fault is developing, supporting maintenance decisions.

**Limitations:** simulated data; real vehicles need calibration across speeds, loads and road types, and baselines should be re-learned per vehicle.